In [12]:
path = "/content/diabetes.csv"

In [13]:
import numpy as np
import pandas as pd

In [14]:
df = pd.read_csv(path)
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [15]:
df.corr()["Outcome"]

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [16]:
x = df.drop(["Outcome"], axis = 1)
y = df["Outcome"]

In [17]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(x_train)

x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)

In [21]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 13.5 MB/s eta 0:00:00


In [22]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import keras_tuner as kt

In [23]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(hp.Choice("units_1", [2**i for i in range(3, 11)]), activation = "relu", input_dim = 8))
  model.add(Dense(hp.Choice("units_2", [2**i for i in range(3, 11)]), activation = "relu"))
  model.add(Dense(1, activation = "sigmoid"))
  model.compile(loss = "binary_crossentropy", optimizer = hp.Choice("optimizer", ["adam", "rmsprop"]), metrics = ["accuracy"])
  return model

In [24]:
tuner = kt.RandomSearch(build_model, objective = "val_accuracy", max_trials = 20, project_name = "project3")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
tuner.search(x_train, y_train, epochs = 10, validation_data = (x_test, y_test))

Trial 20 Complete [00h 00m 05s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 01m 35s


In [26]:
best_model = tuner.get_best_models()[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [27]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,377 (5.38 KB)

 Trainable params: 1,377 (5.38 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
def build_model1(hp):
  model = Sequential()

  model.add(Dense(hp.Int("layer1", min_value = 8, max_value = 1024, step = 8), activation = "relu", input_dim = 8))

  k = 2
  for i in range(hp.Int("num_layers", min_value = 1,max_value = 11)):
    model.add(Dense(hp.Int("layer" + str(k), min_value = 8, max_value = 1024, step = 8), activation = "relu"))
    k+= 1

  model.add(Dense(1, activation = "sigmoid"))

  model.compile(loss = "binary_crossentropy", metrics = ["accuracy"], optimizer = hp.Choice("optimizzer" , ["adam", "rmsprop"]))

  return model

In [37]:
tuner1 = kt.RandomSearch(build_model1, objective = "val_accuracy", max_trials = 15, project_name = "final_tuner1")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
tuner1.search(x_train, y_train, epochs = 10, validation_split = 0.2)

Trial 15 Complete [00h 00m 05s]
val_accuracy: 0.7804877758026123

Best val_accuracy So Far: 0.7967479825019836
Total elapsed time: 00h 02m 10s


In [39]:
best_model = tuner.get_best_models()[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [47]:
pred = best_model.predict(x_test)
print(pred)

func = lambda x: 1 if x > 0.5 else 0

pred_ = np.array([func(i) for i in pred])
print(pred_)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
[[0.39358744]
 [0.24474427]
 [0.23223683]
 [0.3893315 ]
 [0.39878675]
 [0.35391885]
 [0.12340076]
 [0.42387712]
 [0.5489886 ]
 [0.47084412]
 [0.36780635]
 [0.5769604 ]
 [0.4865107 ]
 [0.46022108]
 [0.21273588]
 [0.34766838]
 [0.23231792]
 [0.28972375]
 [0.47594216]
 [0.4112845 ]
 [0.3317064 ]
 [0.20847355]
 [0.43705964]
 [0.1775776 ]
 [0.5312085 ]
 [0.71940124]
 [0.30073497]
 [0.19356215]
 [0.2550599 ]
 [0.3296172 ]
 [0.6790548 ]
 [0.5963513 ]
 [0.603046  ]
 [0.52948856]
 [0.535554  ]
 [0.7257573 ]
 [0.580652  ]
 [0.33234304]
 [0.3820725 ]
 [0.361676  ]
 [0.2461042 ]
 [0.42013752]
 [0.3993073 ]
 [0.40949595]
 [0.35288158]
 [0.53273886]
 [0.638052  ]
 [0.3221636 ]
 [0.3859843 ]
 [0.8560528 ]
 [0.21639588]
 [0.606938  ]
 [0.6379249 ]
 [0.31456208]
 [0.15415537]
 [0.18989299]
 [0.48572832]
 [0.04404917]
 [0.46896482]
 [0.52481693]
 [0.6051592 ]
 [0.3904078 ]
 [0.5690624 ]
 [0.5694474 ]
 [0.15787634]
 [0.37060052]
 [0.18152492]
 [0.5643736 ]
 [0.173917

In [48]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, pred_)

0.7987012987012987